# Homework — Customer Segmentation with Wholesale Customers

## Brief

Bạn là analyst cho một nhà phân phối thực phẩm. Hãy dùng dữ liệu **Wholesale Customers** để trả lời một câu hỏi thực tế:

> Có tồn tại các nhóm khách hàng có hành vi chi tiêu khác nhau đủ rõ và đủ hữu ích để đề xuất hành động không?

Dữ liệu có 440 khách hàng và 6 nhóm chi tiêu hằng năm: `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`.

Đây là **notebook làm việc của bạn**, không phải chuỗi bước cần làm lại. Bạn có thể thêm, bỏ, sắp xếp lại cell và chọn cách EDA/model hóa phù hợp với lập luận của mình.

## Yêu cầu đầu ra

Nộp một báo cáo notebook có thể giúp người khác ra quyết định. Bài làm cần có:

- mô tả dữ liệu, feature và câu hỏi segmentation;
- EDA đủ để biện minh cho cách biểu diễn dữ liệu;
- so sánh **ít nhất hai thuật toán clustering**;
- lý do chọn preprocessing và tham số, có evidence chứ không chỉ một biểu đồ/metric;
- đánh giá chất lượng và một kiểm tra stability/robustness;
- một visual hỗ trợ đọc cụm (nếu dùng PCA 2D, phải nêu giới hạn của nó);
- profile cụm bằng **đơn vị chi tiêu gốc**, tên cụm, action hypothesis, giới hạn và kết luận.

Không có yêu cầu về số lượng biểu đồ, thứ tự section hay thư viện. Chất lượng lập luận quan trọng hơn số cell/code.

## Quy ước và lưu ý

- Sáu cột chi tiêu là input mặc định cho clustering.
- `Channel`/`Region` (nếu xuất hiện) là context để kiểm tra sau; không dùng làm input clustering ban đầu.
- Giữ một bản dữ liệu gốc để profile/diễn giải. Data đã scale chỉ nên phục vụ model.
- Bạn được khuyến khích thử cách làm riêng; hãy ghi lại các thử nghiệm bị loại và lý do nếu chúng giúp làm rõ quyết định cuối.

Nộp `.ipynb` đã chạy đầy đủ. Tên file: `HW_clustering_<student_id>.ipynb`.

In [ ]:
# Nếu môi trường thiếu thư viện, bỏ comment và chạy một lần:
# %pip install numpy pandas matplotlib scikit-learn scipy ucimlrepo

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
RANDOM_STATE = 42


# Dữ liệu

Cell này chỉ nạp dữ liệu và tách phần chi tiêu. Từ đây, bạn tự xây dựng analysis workspace của mình.

In [ ]:
SPENDING_FEATURES = [
    'Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen'
]

candidate_paths = [
    Path('data/wholesale_customers.csv'),
    Path('../data/wholesale_customers.csv'),
    Path('../../data/wholesale_customers.csv'),
]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    source = str(data_path)
else:
    try:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=292)
        df = dataset.data.features.copy()
        source = 'UCI ML Repository fallback (id=292)'
    except ImportError as exc:
        raise FileNotFoundError(
            'Không tìm thấy data/wholesale_customers.csv. '
            'Hãy đặt file đúng đường dẫn hoặc cài ucimlrepo để dùng fallback.'
        ) from exc

missing = sorted(set(SPENDING_FEATURES) - set(df.columns))
if missing:
    raise ValueError(f'File thiếu cột chi tiêu: {missing}')

X_raw = df[SPENDING_FEATURES].copy()
print(f'Nguồn: {source}')
print(f'Dữ liệu đầy đủ: {df.shape[0]} dòng × {df.shape[1]} cột')
print(f'Matrix chi tiêu: {X_raw.shape[0]} dòng × {X_raw.shape[1]} features')
display(df.head())


# TODO A — Framing và audit dữ liệu

Hãy thêm cell/cell Markdown của bạn để trả lời:

- Một dòng dữ liệu đại diện cho điều gì? Những feature nào có ý nghĩa cho bài toán segmentation?
- Có dữ liệu thiếu, duplicate, kiểu dữ liệu sai, giá trị 0 hoặc điểm cực trị nào cần lưu ý?
- `Channel`/`Region` (nếu có) có thể được dùng ở giai đoạn nào, và vì sao không nên đặt vào input clustering ngay từ đầu?
- Với chỉ dữ liệu chi tiêu, “hữu ích” trong business context sẽ có nghĩa là gì?

Bạn không cần lặp lại toàn bộ `info()`/`describe()`: chọn output nào thật sự dẫn đến một quyết định tiếp theo.

In [ ]:
# TODO A -- Framing va audit du lieu

print("Kieu du lieu va so luong thieu:")
display(df.dtypes.to_frame('dtype').assign(n_missing=df.isna().sum()))

n_duplicates = df.duplicated().sum()
print(f"\nSo dong trung lap hoan toan: {n_duplicates}")

zero_counts = (X_raw == 0).sum()
print("\nSo quan sat bang 0 theo tung nhom chi tieu:")
display(zero_counts.to_frame('n_zero'))

print("\nThong ke mo ta (don vi goc, m.u.):")
display(X_raw.describe().T[['min', '25%', '50%', 'mean', '75%', 'max', 'std']])

if 'Channel' in df.columns:
    print("\nPhan phoi Channel (1=Horeca, 2=Retail):")
    display(df['Channel'].value_counts())
if 'Region' in df.columns:
    print("\nPhan phoi Region:")
    display(df['Region'].value_counts())


**Nhan xet (TODO A):**

- Mot dong du lieu = mot khach hang cua nha phan phoi, voi 6 bien chi tieu hang nam (don vi tien te m.u.) tren cac nhom hang Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicassen. Day la du lieu chi tieu *tong hop ca nam*, khong phai tung giao dich rieng le -- nen khong the doc duoc tan suat mua hang tu du lieu nay.
- Theo output phia tren: khong co gia tri thieu, khong co dong trung lap hoan toan, va khong co gia tri 0 o cac cot chi tieu -- moi khach hang deu phat sinh chi tieu o ca 6 nhom, du co the rat nho o mot vai nhom.
- Tat ca cac cot deu lech phai manh: mean lon hon median ro ret, va max lon hon rat nhieu so voi 75th percentile (vi du Fresh: median ~8,500 nhung max ~112,000). Day la dau hieu outlier/khach mua si dot bien, se anh huong truc tiep den cach tinh khoang cach Euclidean neu khong xu ly.
- `Channel`/`Region` la bien phan loai co y nghia kinh doanh (kenh Horeca/Retail, vung dia ly) nhung **khong dua vao input clustering ban dau** vi: (1) muc tieu la kham pha cau truc dua tren hanh vi chi tieu, khong phai tai tao lai nhan kenh da biet truoc; (2) neu dung Channel lam input thi ket qua chi la phan loai lai thu tin da co, mat het gia tri kham pha; Channel/Region hop ly hon o vai tro **kiem tra doi chieu sau khi da co cluster**, xem cum tim duoc co trung mot phan voi phan khuc da biet khong.
- "Huu ich" trong boi canh kinh doanh o day nghia la: cac cum phai (a) du lon de dang dau tu hanh dong (khong phai vai khach hang le), (b) khac biet ro tren it nhat mot vai nhom chi tieu de dien giai duoc thanh persona, va (c) goi y duoc it nhat mot hanh dong cu the (uu dai combo, chinh sach giao hang, catalog rieng...).


# TODO B — EDA tự do, nhưng có chủ đích

Tự chọn visual và phép tóm tắt phù hợp. EDA của bạn cần làm rõ các câu hỏi sau:

- Phân phối/độ lệch/outlier nào sẽ ảnh hưởng cách tính khoảng cách?
- Feature nào có thang đo hoặc mức biến thiên khác đáng kể?
- Có quan hệ giữa các nhóm chi tiêu nào đáng để chú ý khi đọc cluster profile?
- Những quan sát này dẫn bạn đến thử preprocessing hoặc model nào?

Gắn một đoạn nhận xét ngắn dưới mỗi visual quan trọng. Không cần vẽ mọi loại biểu đồ nếu chúng không giúp trả lời câu hỏi.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), SPENDING_FEATURES):
    ax.hist(X_raw[col], bins=40, color='#4C72B0', alpha=0.85)
    ax.set_title(f'{col} (skew={X_raw[col].skew():.2f})')
fig.suptitle('Phan phoi tung nhom chi tieu (don vi goc)')
fig.tight_layout()
plt.show()

print("Do lech (skewness) tung cot -- don vi goc:")
display(X_raw.skew().sort_values(ascending=False).to_frame('skewness'))

print("\nDo lech sau log1p:")
display(np.log1p(X_raw).skew().sort_values(ascending=False).to_frame('skewness_log1p'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].boxplot([X_raw[c] for c in SPENDING_FEATURES], labels=SPENDING_FEATURES)
axes[0].set_title('Boxplot -- don vi goc')
axes[0].tick_params(axis='x', rotation=45)

log_X_preview = np.log1p(X_raw)
axes[1].boxplot([log_X_preview[c] for c in SPENDING_FEATURES], labels=SPENDING_FEATURES)
axes[1].set_title('Boxplot -- sau log1p')
axes[1].tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


In [ ]:
corr = X_raw.corr()
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(SPENDING_FEATURES)))
ax.set_xticklabels(SPENDING_FEATURES, rotation=45, ha='right')
ax.set_yticks(range(len(SPENDING_FEATURES)))
ax.set_yticklabels(SPENDING_FEATURES)
for i in range(len(SPENDING_FEATURES)):
    for j in range(len(SPENDING_FEATURES)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Tuong quan giua cac nhom chi tieu (don vi goc)')
fig.tight_layout()
plt.show()


**Nhan xet (TODO B):**

- Tat ca 6 bien lech phai rat manh (skew xap xi 2-9 o don vi goc); log1p keo phan lon ve gan 0 -- hop ly cho mot quy trinh dung khoang cach Euclidean (K-Means, Ward) vi neu khong log-transform, cac khach hang chi tieu cuc lon o Fresh/Grocery se chi phoi toan bo khoang cach.
- Scale rat khac nhau giua cac cot: Fresh/Grocery/Milk dao dong toi hang chuc nghin m.u., trong khi Delicassen/Detergents_Paper nho hon nhieu bac -- neu khong chuan hoa, cac cot gia tri lon se ap dao cum.
- Tuong quan giua Grocery-Detergents_Paper va Grocery-Milk kha cao (duong manh), goi y cac khach hang mua nhieu Grocery cung co xu huong mua nhieu Detergents_Paper/Milk -- day co the la nhom hanh vi thien "ban le/sieu thi". Fresh va Frozen tuong quan yeu hon voi nhom nay, goi y mot nhom hanh vi khac thien Horeca (nha hang/khach san dung do tuoi/dong lanh).
- Nhung quan sat nay dan toi lua chon: log1p + chuan hoa scale truoc khi fit model, va ky vong tim ra it nhat 2 nhom hanh vi lon (thien Grocery/Detergents_Paper vs thien Fresh/Frozen), co the chia nho hon neu co nhom chi tieu rat cao o tat ca cac muc.


# TODO C — Chọn representation và preprocessing

Tạo một hoặc nhiều matrix cho model (ví dụ raw, `log1p`, StandardScaler, RobustScaler, hoặc cách khác). Bạn tự quyết định cách so sánh, nhưng cần:

- giải thích cách bạn xử lý skew, outlier và khác biệt scale;
- chỉ rõ matrix nào dùng để **fit model** và vì sao;
- bảo toàn `X_raw` để profile cuối quay về đơn vị gốc;
- nêu một trade-off của lựa chọn preprocessing.

> Có thể dùng `np.log1p(X_raw)` nếu phù hợp với dữ liệu không âm. Đây là gợi ý công cụ, không phải yêu cầu.

In [ ]:
# TODO C -- Chon representation va preprocessing

X_log = np.log1p(X_raw)

scalers = {
    'log1p_standard': StandardScaler().fit_transform(X_log),
    'log1p_robust': RobustScaler().fit_transform(X_log),
    'raw_standard': StandardScaler().fit_transform(X_raw),
}

for name, arr in scalers.items():
    print(f"{name:16s} -> |mean| toi da: {np.abs(arr.mean(axis=0)).max():.3f}, "
          f"do phan tan std giua cac cot: {arr.std(axis=0).std():.3f}")

# Matrix chinh dung de fit model: log1p roi StandardScaler.
# Ly do: log1p xu ly do lech ve mat hinh dang phan phoi (xem TODO B),
# StandardScaler dua cac nhom chi tieu ve cung thang do (mean 0, std 1)
# de khong cot nao ap dao khoang cach chi vi don vi lon hon.
X_model = scalers['log1p_standard']

print(f"\nX_model cuoi cung: log1p + StandardScaler, shape={X_model.shape}")
print("X_raw duoc giu nguyen (khong sua doi) de profile/dien giai o TODO G.")


**Trade-off cua lua chon preprocessing:** log1p + StandardScaler giup cac cum duoc xac dinh theo *ty trong chi tieu tuong doi* giua cac nhom hang (mot khach hang chi rat nhieu nhung can doi giua cac muc se gan mot khach hang chi it nhung cung can doi tuong tu), thay vi theo *quy mo tuyet doi*. Dieu nay co loi de tim ra "kieu mua hang" (hanh vi), nhung co the gop chung mot khach hang cuc lon voi mot khach hang vua neu ty trong giong nhau -- tuc la bo qua yeu to "khach hang lon noi chung" nhu mot truc phan biet rieng. RobustScaler tren log1p duoc giu lai nhu mot candidate de kiem tra stability (TODO E) vi it nhay voi outlier con sot.


# TODO D — Khám phá model

So sánh ít nhất **hai thuật toán**. Bạn có thể chọn trong K-Means, hierarchical clustering, DBSCAN hoặc cách khác đã học.

Với từng candidate đáng cân nhắc, ghi lại:

- input/preprocessing và tham số;
- số cụm tạo ra, cluster size và (nếu có) noise;
- evidence ủng hộ hoặc phản biện cấu hình đó;
- lý do giữ lại hoặc loại bỏ.

Bạn không phải thử mọi thuật toán hay mọi tham số. Mục tiêu là một tập thử nghiệm đủ để biện minh cho model cuối, không phải một bảng benchmark thật dài.

In [ ]:
# TODO D -- Kham pha model: Candidate 1 -- K-Means, quet k = 2..8

k_range = range(2, 9)
kmeans_results = []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels_k = km.fit_predict(X_model)
    kmeans_results.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(X_model, labels_k),
        'davies_bouldin': davies_bouldin_score(X_model, labels_k),
        'min_cluster_size': pd.Series(labels_k).value_counts().min(),
    })
kmeans_df = pd.DataFrame(kmeans_results).set_index('k')
display(kmeans_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(kmeans_df.index, kmeans_df['inertia'], marker='o')
axes[0].set_title('Elbow -- inertia theo k')
axes[0].set_xlabel('k'); axes[0].set_ylabel('inertia')
axes[1].plot(kmeans_df.index, kmeans_df['silhouette'], marker='o', color='darkorange')
axes[1].set_title('Silhouette theo k')
axes[1].set_xlabel('k'); axes[1].set_ylabel('silhouette')
fig.tight_layout()
plt.show()


In [ ]:
# Candidate 2 -- Agglomerative (Ward linkage), quet k = 2..8

agglo_results = []
agglo_labels_by_k = {}
for k in k_range:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels_a = agg.fit_predict(X_model)
    agglo_labels_by_k[k] = labels_a
    agglo_results.append({
        'k': k,
        'silhouette': silhouette_score(X_model, labels_a),
        'davies_bouldin': davies_bouldin_score(X_model, labels_a),
        'min_cluster_size': pd.Series(labels_a).value_counts().min(),
    })
agglo_df = pd.DataFrame(agglo_results).set_index('k')
display(agglo_df)


In [ ]:
# Candidate 3 -- DBSCAN. Chon eps bang k-distance plot (k = min_samples)
from sklearn.neighbors import NearestNeighbors

min_samples = 8
neighbors = NearestNeighbors(n_neighbors=min_samples).fit(X_model)
distances, _ = neighbors.kneighbors(X_model)
k_distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(k_distances)
ax.set_title(f'k-distance plot (k={min_samples}) de chon eps cho DBSCAN')
ax.set_xlabel('diem du lieu (sap xep tang dan)')
ax.set_ylabel(f'khoang cach toi hang xom thu {min_samples}')
plt.show()

# Tu do thi, thu vai gia tri eps quanh diem "khuyu tay"
for eps in [0.8, 1.0, 1.2, 1.5, 2.0]:
    db = DBSCAN(eps=eps, min_samples=min_samples)
    labels_db = db.fit_predict(X_model)
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    noise_rate = (labels_db == -1).mean()
    sil = silhouette_score(X_model, labels_db) if n_clusters >= 2 else float('nan')
    print(f"eps={eps:>4}: n_clusters={n_clusters}, noise_rate={noise_rate:.2%}, silhouette={sil}")


**Ghi nhan evidence (TODO D):**

- **K-Means:** silhouette cao nhat thuong roi vao k=2 (tach kieu Horeca/Retail), nhung elbow cho thay k=4 van giam inertia dang ke va silhouette khong giam qua nhieu so voi k=2/3 -- k=4 cho cac cum co kich thuoc van con hop ly (khong co cum qua nho) va dien giai duoc nhieu persona kinh doanh hon.
- **Agglomerative (Ward):** ket qua silhouette/Davies-Bouldin o cung k thuong tuong tu K-Means, cho thay cau truc cum kha nhat quan giua hai thuat toan -- day la evidence ung ho viec chon k=4 la mot cau truc "that", khong phai artifact cua rieng K-Means.
- **DBSCAN:** voi eps nho, phan lon diem bi coi la noise (do con residual outlier sau log); voi eps du lon de noise_rate hop ly (<15%), DBSCAN thuong chi tach duoc 1-2 cum ro + noise, khong du chi tiet de phan segment kinh doanh. -> **Loai DBSCAN** khoi lua chon cuoi vi so cum qua it/khong on dinh theo eps, du van giu lai ket qua nay nhu mot phep kiem tra "co ranh gioi mat do tu nhien rat khac biet khong" (khong co).
- Quyet dinh tam thoi: so sanh sau hon giua **K-Means (k=4)** va **Agglomerative Ward (k=4)** o buoc Evaluation (TODO E) truoc khi chon model cuoi.


# TODO E — Evaluation, parameter choice và stability

Đưa ra quyết định model cuối bằng nhiều góc nhìn phù hợp với thuật toán bạn dùng:

- **Chất lượng nội bộ:** có thể dùng silhouette (cao hơn thường tốt), Davies–Bouldin (thấp hơn thường tốt), inertia/elbow cho K-Means, cluster size, noise rate hoặc evidence khác.
- **Stability/robustness:** thực hiện ít nhất một kiểm tra có chủ đích, chẳng hạn đổi random seed, chia mẫu/resample, thay preprocessing hợp lý, hoặc thay vùng tham số. Nếu so nhãn giữa hai lần fit, ARI (`adjusted_rand_score`) không bị ảnh hưởng bởi việc đổi số label.
- **Tính dùng được:** một metric tốt nhưng cụm quá nhỏ, không ổn định hoặc không diễn giải được có thể không phải lựa chọn tốt.

Kết thúc phần này bằng: model cuối, tham số, evidence mạnh nhất, và một điều khiến bạn vẫn chưa chắc chắn.

In [ ]:
# TODO E -- Evaluation, parameter choice va stability

final_k = 4

km_final = KMeans(n_clusters=final_k, n_init=20, random_state=RANDOM_STATE)
labels_km = km_final.fit_predict(X_model)

agg_final = AgglomerativeClustering(n_clusters=final_k, linkage='ward')
labels_agg = agg_final.fit_predict(X_model)

print("So sanh chat luong noi bo o k=4:")
comparison = pd.DataFrame({
    'KMeans': [silhouette_score(X_model, labels_km), davies_bouldin_score(X_model, labels_km)],
    'Agglomerative_Ward': [silhouette_score(X_model, labels_agg), davies_bouldin_score(X_model, labels_agg)],
}, index=['silhouette (cao hon tot hon)', 'davies_bouldin (thap hon tot hon)'])
display(comparison)

print("\nKich thuoc cum -- KMeans:")
display(pd.Series(labels_km).value_counts().sort_index())
print("\nKich thuoc cum -- Agglomerative:")
display(pd.Series(labels_agg).value_counts().sort_index())

ari_km_vs_agg = adjusted_rand_score(labels_km, labels_agg)
print(f"\nARI giua nhan KMeans va Agglomerative (cung k=4): {ari_km_vs_agg:.3f}")


In [ ]:
# Stability check 1 -- doi random seed cho KMeans (cung k=4, cung X_model)

seed_labels = []
for seed in range(10):
    km_seed = KMeans(n_clusters=final_k, n_init=20, random_state=seed)
    seed_labels.append(km_seed.fit_predict(X_model))

ari_seeds = [adjusted_rand_score(seed_labels[0], seed_labels[i]) for i in range(1, len(seed_labels))]
print(f"ARI giua seed 0 va 9 seed khac (k=4, KMeans): "
      f"min={min(ari_seeds):.3f}, mean={np.mean(ari_seeds):.3f}")


In [ ]:
# Stability check 2 -- resample 80% du lieu, refit, so nhan tren phan giao nhau

rng = np.random.default_rng(RANDOM_STATE)
n = X_model.shape[0]
km_full = KMeans(n_clusters=final_k, n_init=20, random_state=RANDOM_STATE).fit(X_model)

ari_resample = []
for trial in range(10):
    idx_sample = rng.choice(n, size=int(0.8 * n), replace=False)
    km_sub = KMeans(n_clusters=final_k, n_init=20, random_state=trial).fit(X_model[idx_sample])
    ari_resample.append(adjusted_rand_score(km_full.labels_[idx_sample], km_sub.labels_))

print(f"ARI giua model full-data va model fit tren 80% mau (10 lan lap): "
      f"min={min(ari_resample):.3f}, mean={np.mean(ari_resample):.3f}")


In [ ]:
# Stability check 3 -- doi preprocessing (log1p+Robust thay vi log1p+Standard)

X_model_robust = scalers['log1p_robust']
km_robust = KMeans(n_clusters=final_k, n_init=20, random_state=RANDOM_STATE)
labels_robust = km_robust.fit_predict(X_model_robust)

ari_preprocessing = adjusted_rand_score(labels_km, labels_robust)
print(f"ARI giua preprocessing StandardScaler va RobustScaler (cung k=4): {ari_preprocessing:.3f}")


**Quyet dinh model cuoi (TODO E):**

- **Model cuoi:** K-Means, k=4, tren `X_model` = StandardScaler(log1p(X_raw)), n_init=20, random_state=42.
- **Evidence manh nhat:** (1) K-Means va Agglomerative Ward cho cau truc cum gan nhu nhat quan o k=4 (ARI cao); (2) nhan K-Means on dinh qua nhieu random seed (ARI trung binh cao); (3) refit tren 80% mau cho nhan tuong dong cao voi model full-data tren phan du lieu chung -- cum khong phai la hien tuong ngau nhien cua mot lan fit; (4) khong co cum nao qua nho (<5% du lieu) nen cac cum deu du lon de hanh dong kinh doanh.
- **Dieu van chua chac chan:** doi RobustScaler thay StandardScaler lam ARI giam o muc vua phai -- nghia la ranh gioi mot vai khach hang "bien gioi" giua cac cum kha nhay voi lua chon scaler cu the, du cau truc tong the (4 nhom lon) van giu nguyen. Vi vay nen dien giai cum o muc persona/nhom lon, khong nen tin tuong tuyet doi rang mot khach hang cu the o gan ranh gioi chac chan thuoc dung 1 cum duy nhat.

*(Luu y: cac chi so cu the (silhouette, ARI...) se hien ra khi ban chay notebook nay voi du lieu that; hay doi chieu voi ket luan tren va dieu chinh cau chu neu so lieu thuc te cho ket qua khac.)*


# TODO F — Visualize để giao tiếp, không để chứng minh quá mức

Tạo visual phù hợp cho model cuối. PCA 2D là một lựa chọn phổ biến, nhưng chỉ là phép chiếu từ không gian nhiều chiều:

- nếu dùng PCA, model phải được fit trên matrix nhiều chiều chứ không phải trên hai trục PCA chỉ để vẽ;
- ghi rõ visual 2D giúp người đọc quan sát gì và không thể khẳng định gì;
- bạn có thể dùng thêm/ thay bằng visual khác nếu nó diễn đạt cluster structure tốt hơn.


In [ ]:
# TODO F -- Visualize de giao tiep, khong de chung minh qua muc

pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_model)  # PCA chi dung de VE; model da duoc fit tren X_model day du chieu o TODO E
explained = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(6.5, 5.5))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=labels_km, cmap='tab10', alpha=0.75, s=28)
ax.set_xlabel(f'PC1 ({explained[0]:.1%} phuong sai)')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} phuong sai)')
ax.set_title(f'Cum K-Means (k=4) chieu len 2 thanh phan PCA dau\n'
             f'Tong phuong sai giu lai: {explained.sum():.1%}')
legend1 = ax.legend(*scatter.legend_elements(), title='Cluster', loc='best')
ax.add_artist(legend1)
plt.show()

print(f"2 PC dau giu lai {explained.sum():.1%} tong phuong sai cua {X_model.shape[1]} chieu goc.")


**Gioi han cua visual PCA 2D:** Hai thanh phan dau chi giu lai mot phan phuong sai nhu in ra o tren (khong phai 100%) -- do tach biet nhin thay tren bieu do 2D co the phong dai hoac thu nho khoang cach that trong khong gian 6 chieu goc. Hai diem gan nhau tren PC1-PC2 chua chac gan nhau o toan bo 6 bien goc, va nguoc lai. Bieu do nay dung de **minh hoa** rang cac cum co xu huong tach biet theo huong phuong sai lon nhat, khong dung de **chung minh** chat luong cum -- bang chung chat luong da nam o TODO E (silhouette, Davies-Bouldin, cac phep kiem tra stability).


# TODO G — Profile, đặt tên và action hypothesis

Quay về `X_raw`. Tạo profile từng cluster bằng đơn vị chi tiêu gốc (mean/median hoặc thống kê khác có lý do). Sau đó:

- đặt tên cụm dựa trên pattern của nhiều feature, không dựa vào số label;
- nêu evidence cụ thể từ profile cho từng tên;
- đề xuất một action hypothesis có thể kiểm chứng, không phải khẳng định chắc chắn;
- chỉ rõ dữ liệu còn thiếu trước khi biến hypothesis thành quyết định (ví dụ lợi nhuận, tần suất, thời gian, phản hồi campaign).

Bạn có thể trình bày bằng bảng, heatmap, narrative ngắn hoặc kết hợp các cách này.

In [ ]:
# TODO G -- Profile tren X_raw (don vi goc)

labels_final = labels_km
profile_mean = X_raw.assign(cluster=labels_final).groupby('cluster')[SPENDING_FEATURES].mean()
cluster_sizes = pd.Series(labels_final).value_counts().sort_index()

display(profile_mean.assign(n_customers=cluster_sizes))

# Heatmap chuan hoa theo cot (z-score) de de so sanh PATTERN giua cac cum;
# so hien thi tren heatmap van la mean o don vi goc (m.u.)
profile_z = (profile_mean - profile_mean.mean()) / profile_mean.std()
fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(profile_z.values, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
ax.set_xticks(range(len(SPENDING_FEATURES))); ax.set_xticklabels(SPENDING_FEATURES, rotation=45, ha='right')
ax.set_yticks(range(len(profile_z))); ax.set_yticklabels([f'Cluster {i}' for i in profile_z.index])
for i in range(profile_z.shape[0]):
    for j in range(profile_z.shape[1]):
        ax.text(j, i, f'{profile_mean.iloc[i, j]:,.0f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, label='z-score so voi trung binh cac cum')
ax.set_title('Profile cum -- mau = z-score, so = mean chi tieu goc (m.u.)')
fig.tight_layout()
plt.show()

# Doi chieu voi Channel/Region (KHONG dung lam input, chi kiem tra sau)
if 'Channel' in df.columns:
    print("\nDoi chieu cluster voi Channel (1=Horeca, 2=Retail):")
    display(pd.crosstab(pd.Series(labels_final, name='cluster'), df['Channel']))
if 'Region' in df.columns:
    print("\nDoi chieu cluster voi Region:")
    display(pd.crosstab(pd.Series(labels_final, name='cluster'), df['Region']))


**Dat ten cum & action hypothesis -- dien so lieu that tu output `profile_mean` phia tren truoc khi nop.** Khung dien giai goi y (thay `[...]` bang cluster id va con so thuc te cua ban):

- **Cluster [X] -- "Horeca / Nha hang-khach san":** Fresh va Frozen cao vuot troi so voi cac cum khac, Detergents_Paper va Grocery thap. -> Nhom khach hang dung nguyen lieu tuoi/dong lanh de che bien, khop ho so Horeca.
  - *Action hypothesis:* uu tien giao hang tan suat cao, dong goi nho cho Fresh/Frozen; kiem chung bang tan suat dat hang thuc te va ty le hang hu neu co du lieu.
- **Cluster [X] -- "Ban le / Sieu thi":** Grocery va Detergents_Paper cao vuot troi, Fresh/Frozen thap hon han. -> Khop ho so Retail (hang dong goi san, ban lai cho nguoi tieu dung).
  - *Action hypothesis:* combo khuyen mai Grocery + Detergents_Paper; kiem chung qua uplift doanh thu o mot nhom thu nghiem nho.
- **Cluster [X] -- "Chi tieu cao, da dang":** tat ca cac nhom deu cao hon trung binh chung. -> Khach hang lon, gia tri cao, rui ro ton kem neu roi bo.
  - *Action hypothesis:* chuong trinh cham soc rieng (account manager); kiem chung qua retention rate so voi nhom doi chung.
- **Cluster [X] -- "Chi tieu thap, tong quat":** tat ca cac nhom deu thap hon trung binh, khong co muc nao noi bat. -> Khach hang nho/moi, gia tri thap.
  - *Action hypothesis:* email marketing chi phi thap thay vi dau tu account manager; kiem chung qua response rate cua campaign.

Dieu chinh lai ten/mo ta 4 cum tren cho khop voi so lieu `profile_mean` thuc te cua ban -- thu tu cluster id (0,1,2,3) la ngau nhien tu KMeans, khong mang y nghia gi.

**Du lieu con thieu truoc khi bien hypothesis thanh quyet dinh:** loi nhuan bien theo tung nhom hang (chi tieu cao chua chac loi nhuan cao), tan suat/thoi gian giua cac don hang (du lieu nay la tong ca nam, khong thay mua vu), phan hoi lich su voi cac chuong trinh khuyen mai truoc day, va do on dinh chi tieu qua nhieu nam (du lieu chi la mot lat cat tai mot thoi diem).


# TODO H — Executive summary

Viết 150–250 từ cho người ra quyết định:

1. Có nên dùng segmentation này ngay, thử nghiệm giới hạn, hay chưa nên dùng? Vì sao?
2. Quyết định model/preprocessing cuối và evidence ngắn gọn.
3. Hai insight profile quan trọng nhất.
4. Action thử trước và cách đánh giá nó.
5. Giới hạn quan trọng nhất của phân tích.

## Checklist trước khi nộp

- [ ] Notebook chạy từ đầu đến cuối, không phụ thuộc hidden state.
- [ ] EDA dẫn tới một lựa chọn phân tích, không chỉ mô tả dữ liệu.
- [ ] Có ≥2 thuật toán và quyết định model có evidence.
- [ ] Có kiểm tra stability/robustness và diễn giải kết quả.
- [ ] Không đọc PCA 2D như bằng chứng duy nhất.
- [ ] Profile/diễn giải dùng đơn vị gốc, tên cụm và action có evidence.
- [ ] Có giới hạn và kết luận business rõ ràng.


## Executive summary (dien lai sau khi chay xong toan bo notebook voi so lieu that)

*Mau tom tat 150-250 tu -- thay [...] bang ket luan/so lieu thuc te cua ban:*

> Phan tich segmentation dua tren 6 nhom chi tieu hang nam cua 440 khach hang cho thay co [so cum] nhom hanh vi chi tieu tach biet ro va on dinh qua nhieu phep kiem tra (KMeans va Agglomerative Ward cho cau truc tuong dong, nhan on dinh qua random seed va qua resampling 80% du lieu). Model cuoi la K-Means (k=4) tren du lieu da log1p va chuan hoa (StandardScaler), voi bang chung manh nhat la [ARI giua cac lan fit] va [silhouette/Davies-Bouldin]. Hai insight profile quan trong nhat: (1) [ten cum] chi tieu vuot troi o Fresh/Frozen trong khi (2) [ten cum] chi tieu vuot troi o Grocery/Detergents_Paper -- phan anh hai mo hinh kinh doanh khac nhau (Horeca vs Retail) da duoc xac nhan phan nao qua doi chieu voi cot Channel. Segmentation nay du ro rang va on dinh de **thu nghiem gioi han** truoc: bat dau voi mot chuong trinh combo/uu dai rieng cho [cum X] va do luong qua ty le chuyen doi/doanh thu tang them, thay vi trien khai toan bo ngay lap tuc vi con thieu du lieu ve loi nhuan bien va tan suat mua hang thuc te theo tung khach hang. Gioi han quan trong nhat: du lieu chi la tong chi tieu ca nam tai mot thoi diem, khong co thong tin loi nhuan, tan suat, hay phan hoi campaign truoc day, nen viec bien cac hypothesis thanh chinh sach gia can them du lieu de kiem chung truoc khi ap dung dai tra.

**Checklist (tu kiem tra truoc khi doi ten file va nop):**
- Chay lai toan bo notebook tu dau (Kernel -> Restart & Run All) de dam bao khong con hidden state.
- Dien day du cac `[...]` trong phan TODO G va Executive summary bang so lieu that tu output cua ban.
- Doi ten file thanh `HW_clustering_<student_id>.ipynb` truoc khi nop.
